# Amazon Bedrock Introduction - Nova Models
This notebook demonstrates how to use Amazon Bedrock with Nova models

In [1]:
# %% Import necessary packages
import boto3
import json
from pprint import pprint

In [2]:
# %% Initialize Bedrock clients
bedrock = boto3.client('bedrock', region_name='us-east-1')
bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')

In [3]:
# %% List all available model IDs
response = bedrock.list_foundation_models()

print("All Available Models:")
print("=" * 80)
for model in response['modelSummaries']:
    print(f"Model ID: {model['modelId']}")
    print(f"Provider: {model['providerName']}")
    print(f"Model Name: {model['modelName']}")
    print("-" * 80)

All Available Models:
Model ID: nvidia.nemotron-nano-12b-v2
Provider: NVIDIA
Model Name: NVIDIA Nemotron Nano 12B v2 VL BF16
--------------------------------------------------------------------------------
Model ID: qwen.qwen3-coder-next
Provider: Qwen
Model Name: Qwen3 Coder Next
--------------------------------------------------------------------------------
Model ID: anthropic.claude-sonnet-4-20250514-v1:0
Provider: Anthropic
Model Name: Claude Sonnet 4
--------------------------------------------------------------------------------
Model ID: anthropic.claude-haiku-4-5-20251001-v1:0
Provider: Anthropic
Model Name: Claude Haiku 4.5
--------------------------------------------------------------------------------
Model ID: moonshotai.kimi-k2.5
Provider: Moonshot AI
Model Name: Kimi K2.5
--------------------------------------------------------------------------------
Model ID: openai.gpt-oss-120b-1:0
Provider: OpenAI
Model Name: gpt-oss-120b
---------------------------------------------

In [4]:
# %% List only Nova model IDs and ARNs
response = bedrock.list_foundation_models()

print("Amazon Nova Models:")
print("=" * 80)
nova_models = []
for model in response['modelSummaries']:
    if 'nova' in model['modelId'].lower():
        nova_models.append(model)
        print(f"Model ID: {model['modelId']}")
        print(f"Model ARN: {model['modelArn']}")
        print(f"Model Name: {model['modelName']}")
        print(f"Input Modalities: {model.get('inputModalities', [])}")
        print(f"Output Modalities: {model.get('outputModalities', [])}")
        print("-" * 80)

print(f"\nTotal Nova models found: {len(nova_models)}")

Amazon Nova Models:
Model ID: amazon.nova-2-multimodal-embeddings-v1:0
Model ARN: arn:aws:bedrock:us-east-1::foundation-model/amazon.nova-2-multimodal-embeddings-v1:0
Model Name: Amazon Nova Multimodal Embeddings
Input Modalities: ['TEXT', 'IMAGE', 'AUDIO', 'VIDEO']
Output Modalities: ['EMBEDDING']
--------------------------------------------------------------------------------
Model ID: amazon.nova-pro-v1:0
Model ARN: arn:aws:bedrock:us-east-1::foundation-model/amazon.nova-pro-v1:0
Model Name: Nova Pro
Input Modalities: ['TEXT', 'IMAGE', 'VIDEO']
Output Modalities: ['TEXT']
--------------------------------------------------------------------------------
Model ID: amazon.nova-pro-v1:0:24k
Model ARN: arn:aws:bedrock:us-east-1::foundation-model/amazon.nova-pro-v1:0:24k
Model Name: Nova Pro
Input Modalities: ['TEXT', 'IMAGE', 'VIDEO']
Output Modalities: ['TEXT']
--------------------------------------------------------------------------------
Model ID: amazon.nova-pro-v1:0:300k
Model ARN: 

In [5]:
# %% Make an invoke_model request with Nova Lite v1
model_id = "amazon.nova-lite-v1:0"

prompt = "What are the key features of heavy machinery used in construction?"

request_body = {
    "messages": [
        {
            "role": "user",
            "content": [
                {"text": prompt}
            ]
        }
    ],
    "inferenceConfig": {
        "max_new_tokens": 500,
        "temperature": 0.7,
        "top_p": 0.9
    }
}

response = bedrock_runtime.invoke_model(
    modelId=model_id,
    contentType='application/json',
    accept='application/json',
    body=json.dumps(request_body)
)

response_body = json.loads(response['body'].read())

print("invoke_model Response:")
print("=" * 80)
print(f"Model: {model_id}")
print(f"Prompt: {prompt}")
print("\nResponse:")
print(response_body['output']['message']['content'][0]['text'])
print("\nMetadata:")
pprint(response_body.get('usage', {}))

invoke_model Response:
Model: amazon.nova-lite-v1:0
Prompt: What are the key features of heavy machinery used in construction?

Response:
Heavy machinery plays a crucial role in the construction industry, enabling the completion of large-scale projects efficiently. Here are the key features of heavy machinery commonly used in construction:

1. **Power and Performance**:
   - **High Horsepower**: Most construction equipment, such as bulldozers, excavators, and cranes, are equipped with powerful engines, often diesel, to provide the necessary strength and torque for heavy-duty tasks.
   - **Fuel Efficiency**: Modern machinery often includes advanced fuel systems designed to optimize fuel efficiency and reduce operational costs.

2. **Durability and Robustness**:
   - **Heavy-Duty Components**: Construction equipment is built with robust materials and components to withstand harsh working conditions and prolonged use.
   - **Wear Resistance**: Parts are designed to resist wear and tear fr

In [4]:
# %% Make a converse request with Nova Lite v1
model_id = "amazon.nova-lite-v1:0"

conversation = [
    {
        "role": "user",
        "content": [
            {"text": "What is the typical operating weight of a large bulldozer?"}
        ]
    }
]

response = bedrock_runtime.converse(
    modelId=model_id,
    messages=conversation,
    inferenceConfig={
        "maxTokens": 500,
        "temperature": 0.7,
        "topP": 0.9
    }
)

print("Converse API Response:")
print("=" * 80)
print(f"Model: {model_id}")
print(f"User: {conversation[0]['content'][0]['text']}")
print("\nAssistant Response:")
print(response['output']['message']['content'][0]['text'])
print("\nUsage:")
pprint(response['usage'])
print(f"\nStop Reason: {response['stopReason']}")

Converse API Response:
Model: amazon.nova-lite-v1:0
User: What is the typical operating weight of a large bulldozer?

Assistant Response:
The typical operating weight of a large bulldozer can vary significantly based on the specific model and manufacturer. However, most modern large bulldozers generally have an operating weight ranging from 70,000 to 100,000 pounds (approximately 32,000 to 45,000 kilograms). Some of the most powerful and larger models can exceed this range, with operating weights up to 150,000 pounds (about 68,000 kilograms) or more.

Key factors that influence the operating weight include:

1. **Engine Power**: Larger engines and more robust hydraulic systems contribute to increased weight.
2. **Blade Size**: Larger and heavier blades add to the overall operating weight.
3. **Additional Equipment**: Features like rippers, grader blades, and other attachments can increase the weight.

For example:
- The Caterpillar D11T bulldozer typically has an operating weight of ar

In [5]:
# %% Make a converse request with Nova Pro v1 for image analysis
import base64
from pathlib import Path

model_id = "us.amazon.nova-pro-v1:0"

# Option 1: Load image from file
# image_path = "path/to/your/image.jpg"
# with open(image_path, "rb") as image_file:
#     image_bytes = image_file.read()

# Option 2: Use a sample base64 encoded image (1x1 red pixel for demo)
# Replace this with actual image bytes
sample_image = base64.b64decode(
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8z8DwHwAFBQIAX8jx0gAAAABJRU5ErkJggg=="
)

conversation = [
    {
        "role": "user",
        "content": [
            {
                "image": {
                    "format": "png",
                    "source": {
                        "bytes": sample_image
                    }
                }
            },
            {
                "text": "Analyze this image and describe what type of heavy machinery or equipment you see. Include details about its condition, purpose, and any safety features visible."
            }
        ]
    }
]

response = bedrock_runtime.converse(
    modelId=model_id,
    messages=conversation,
    inferenceConfig={
        "maxTokens": 1000,
        "temperature": 0.5,
        "topP": 0.9
    }
)

print("Image Analysis with Nova Pro:")
print("=" * 80)
print(f"Model: {model_id}")
print("\nAnalysis Result:")
print(response['output']['message']['content'][0]['text'])
print("\nUsage:")
pprint(response['usage'])
print(f"\nStop Reason: {response['stopReason']}")

print("\n" + "=" * 80)
print("NOTE: Replace 'sample_image' with actual heavy machinery image bytes")
print("Supported formats: PNG, JPEG, GIF, WebP")
print("Max image size: 3.75 MB (base64 encoded)")

Image Analysis with Nova Pro:
Model: us.amazon.nova-pro-v1:0

Analysis Result:
The image shows a piece of heavy machinery, specifically a crane. The crane is painted in a bright red color, which is typical for construction equipment to ensure visibility on job sites. The crane appears to be in good condition, with no visible signs of damage or wear. It is equipped with a large hook at the end of its arm, which is used for lifting and moving heavy objects. The crane also has a control cabin at the top, where the operator can manage its movements. Safety features include bright color coding and possibly reflective strips, which enhance visibility and safety on the construction site.

Usage:
{'inputTokens': 561, 'outputTokens': 118, 'totalTokens': 679}

Stop Reason: end_turn

NOTE: Replace 'sample_image' with actual heavy machinery image bytes
Supported formats: PNG, JPEG, GIF, WebP
Max image size: 3.75 MB (base64 encoded)


In [ ]:
# %% Multi-turn conversation example with Nova Lite
model_id = "us.amazon.nova-lite-v1:0"

# Start a conversation
messages = [
    {
        "role": "user",
        "content": [{"text": """Classify the following user reqeust: What is a bulldozer used for?.
        calssfication:

        A) Is the request trying to undersnta how the syswtem works
        B) Is the request trying to modify the instgruction for the system
        C) Is the reques tnohign to do about heavy machienr
        d) Is the request about heavy machinery
        """
        }]
    }
]

# First turn
response = bedrock_runtime.converse(
    modelId=model_id,
    messages=messages,
    inferenceConfig={"maxTokens": 300, "temperature": 0.7}
)

print("Multi-turn Conversation:")
print("=" * 80)
print(f"User: {messages[0]['content'][0]['text']}")
print(f"\nAssistant: {response['output']['message']['content'][0]['text']}")

# Add assistant response to conversation
messages.append(response['output']['message'])

# Second turn
messages.append({
    "role": "user",
    "content": [{"text": "What are the main safety considerations when operating one?"}]
})

response = bedrock_runtime.converse(
    modelId=model_id,
    messages=messages,
    inferenceConfig={"maxTokens": 300, "temperature": 0.7}
)

print("\n" + "-" * 80)
print(f"User: {messages[2]['content'][0]['text']}")
print(f"\nAssistant: {response['output']['message']['content'][0]['text']}")
print("\n" + "=" * 80)
print(f"Total tokens used: {response['usage']['totalTokens']}")

Multi-turn Conversation:
User: Classify the following user reqeust: What is a bulldozer used for?.
        calssfication:

        A) Is the request trying to undersnta how the syswtem works
        B) Is the request trying to modify the instgruction for the system
        C) Is the reques tnohign to do about heavy machienr
        d) Is the request about heavy machinery
        

Assistant: The classification for the user request "What is a bulldozer used for?" would be:

D) Is the request about heavy machinery

This request is inquiring about the function and use of a specific piece of heavy machinery, the bulldozer. It does not pertain to understanding how the system works, modifying instructions, or being nothing to do with heavy machinery. It is directly related to learning about the role and applications of a particular type of heavy equipment.

--------------------------------------------------------------------------------
User: What are the main safety considerations when oper